# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a walkthrough for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
- Published: 2026-07-11
- License: [ODC-BY-1.0](https://opendatacommons.org/licenses/by/1-0/)
- Keywords: protein abundance, modification analysis, peptide sequences, molecular weight, human mast cells, extracellular vesicles


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset (metadata + record set structure)
dataset = mlc.Dataset(url)
print(f"Loaded dataset: {dataset.metadata.name}\nDescription: {dataset.metadata.description}\n")


## 2. Data Overview
Review available **Record Sets** and their fields. All entities are referenced by their `@id` attributes as defined in the Croissant schema.

In [ ]:
print("# Available Record Sets (cr:RecordSet):\n")
record_set_ids = []
record_set_map = {}
for rs in dataset.metadata.record_sets:
    print(f"- @id: {rs.id}")
    record_set_ids.append(rs.id)
    record_set_map[rs.id] = rs
    print(f"  Name: {rs.name if hasattr(rs, 'name') else '(no name)'}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else '(no description)'}\n")
    if hasattr(rs, 'fields'):
        print("  Fields/Columns:")
        for field in rs.fields:
            print(f"    - @id: {field.id} | name: {getattr(field, 'name', '')} | type: {getattr(field, 'data_type', '')}")
    print()

# For simplicity, enumerate records for the first record set found (most datasets have at least one main record set)
if record_set_ids:
    print(f"\nFirst record set @id: {record_set_ids[0]}\nSample records (first 2):")
    for i, record in enumerate(dataset.records(record_set=record_set_ids[0])):
        print(record)
        if i >= 1:
            break


## 3. Data Extraction
Extract each record set as a Pandas DataFrame using the record set `@id` reference.

In [ ]:
# Use all available record set @ids
record_sets = record_set_ids.copy()
dataframes = {}
for rsid in record_sets:
    print(f"Loading records for RecordSet: '{rsid}' ...")
    records = list(dataset.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)
    print(f"  Loaded shape: {dataframes[rsid].shape}")

# Print available columns for the main record set
main_record_set_id = record_sets[0]  # Choose the first for demonstration
print(f"\nColumns in main RecordSet ({main_record_set_id}):\n{dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate EDA using one numeric field and one categorical field, both referenced by their `@id` (as per Croissant schema).

In [ ]:
# For this example, pick a common numeric field and group field (update @id as appropriate for actual dataset)
# Let's inspect available columns to choose one (update if your dataset differs)
print(f"Available columns in {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())

# Example picks: (Replace below with exact @id from your data dictionary)
# Suppose '@id': 'cr:field/abundance', data-type float
numeric_field = 'cr:field/abundance'  # Example, replace with actual field id

# Categorical/grouping field (e.g., 'cr:field/sample_name')
group_field = 'cr:field/sample_name'  # Example, replace with actual field id

# Filter out records with abundance > threshold
threshold = 10
df = dataframes[main_record_set_id].copy()
if numeric_field in df.columns:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (total: {len(filtered_df)}):")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    # Group by group_field
    if group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print(f"Field '{numeric_field}' not found. Available fields: {df.columns.tolist()}")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

- Example: Distribution of the numeric field, grouped by sample name.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=50, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group
    if group_field in df.columns:
        plt.figure(figsize=(12,6))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.xticks(rotation=90)
        plt.title(f'Boxplot of {numeric_field} by {group_field}')
        plt.show()


## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load, explore, and analyze the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json), referencing all entities by their Croissant `@id`s. You can adapt this workflow to any Croissant-compatible dataset for efficient, reproducible data loading and analysis.

**Key findings:**
- The dataset contains record sets for mass spectrometry protein abundance analysis.
- Data can be easily filtered, normalized, and grouped using Pandas, thanks to a clear schema mapping via `@id`.
- Visual EDA enables quick assessment of distributions and sample relationships in high-throughput datasets.

For more detailed schema documentation or downstream ML use cases, consult the [Croissant documentation](https://mlcommons.org/croissant/) or the [dataset landing page](https://sen.science/doi/10.71728/senscience.vq4a-28xa).
